# LightshowAI XAS notebook

This notebook shows how to fetch a crystal structure from the Materials Project, download the required LightshowAI model checkpoints automatically, and run a simple XAS prediction for a chosen absorbing element and theory combination.

## Before you run it

1. Install the required Python packages in your environment.
2. Create a Materials Project API key.
3. Set that key in the configuration cell below, or export it as an environment variable named `MP_API_KEY`.

## How to generate a Materials Project API key

1. Create or sign in to your Materials Project account.
2. Open your dashboard or API settings page.
3. Generate a new API key.
4. Copy the key and paste it into the configuration cell, or set it in your shell with `export MP_API_KEY=...` before starting Jupyter.

## Expected outputs

- A downloaded `model_checkpoints/` folder in the notebook working directory.
- A structure summary for the selected Materials Project material.
- A dictionary-like prediction output for the selected absorbing element.


## Environment setup

This notebook was developed for **Python 3.11.x** and the package versions listed below.

### Recommended workflow

1. Create and activate a fresh Python 3.11 environment.
2. Install the required packages.
3. Restart the notebook kernel after installation.

If you already maintain a `requirements.txt` file in your project, using that file is the cleanest option.
The code cell below also includes a self-contained `%pip` command for notebook users who want to install dependencies directly from Jupyter.


In [1]:
# --- Optional notebook dependency installation ---
# This notebook expects Python 3.11.x.
#
# Best practice:
#   1. Create a fresh Python 3.11 environment outside the notebook.
#   2. Install dependencies there.
#   3. Start Jupyter from that environment.
#
# If you need to install packages from inside the notebook, uncomment ONE of the
# options below and run it once, then restart the kernel.

# Option A: install from a requirements file kept with the project
# %pip install -r requirements.txt

# Option B: self-contained install with explicit package versions
# %pip install \
#   "crystal_toolkit>=2024.10.22" \
#   "dgl==2.1.0" \
#   "dash==3.4.0" \
#   "lightning>=2.4.0" \
#   "matgl==0.8.5" \
#   "mp-api==0.41.2" \
#   "numpy<=1.26.4" \
#   "pymatgen>=2024.3.1,<=2024.11.13" \
#   "torch==2.2.0" \
#   "torchdata==0.7.1" \
#   "gunicorn>=23.0.0" \
#   "gevent>=25.5.0" \
#   "emmet-core==0.84.3rc0" \
#   "tiled[all]" \
#   "redis>=5.0.0"

REQUIRED_PACKAGES = [
    "crystal_toolkit>=2024.10.22",
    "dgl==2.1.0",
    "dash==3.4.0",
    "lightning>=2.4.0",
    "matgl==0.8.5",
    "mp-api==0.41.2",
    "numpy<=1.26.4",
    "pymatgen>=2024.3.1,<=2024.11.13",
    "torch==2.2.0",
    "torchdata==0.7.1",
    "gunicorn>=23.0.0",
    "gevent>=25.5.0",
    "emmet-core==0.84.3rc0",
    "tiled[all]",
    "redis>=5.0.0",
]

print("Notebook target Python version: 3.11.x")
print("Pinned/required packages for this notebook:")
for pkg in REQUIRED_PACKAGES:
    print(f"  - {pkg}")


Notebook target Python version: 3.11.x
Pinned/required packages for this notebook:
  - crystal_toolkit>=2024.10.22
  - dgl==2.1.0
  - dash==3.4.0
  - lightning>=2.4.0
  - matgl==0.8.5
  - mp-api==0.41.2
  - numpy<=1.26.4
  - pymatgen>=2024.3.1,<=2024.11.13
  - torch==2.2.0
  - torchdata==0.7.1
  - gunicorn>=23.0.0
  - gevent>=25.5.0
  - emmet-core==0.84.3rc0
  - tiled[all]
  - redis>=5.0.0


In [2]:
# --- Optional environment validation ---
# Run this cell after installation to confirm the notebook kernel is using
# a compatible Python version and to print installed package versions.

import sys
import importlib.metadata as md

print(f"Running Python: {sys.version.split()[0]}")
if not (sys.version_info.major == 3 and sys.version_info.minor == 11):
    print("WARNING: This notebook was tested with Python 3.11.x")

PACKAGE_NAME_MAP = {
    "crystal_toolkit": "crystal-toolkit",
    "dgl": "dgl",
    "dash": "dash",
    "lightning": "lightning",
    "matgl": "matgl",
    "mp_api": "mp-api",
    "numpy": "numpy",
    "pymatgen": "pymatgen",
    "torch": "torch",
    "torchdata": "torchdata",
    "gunicorn": "gunicorn",
    "gevent": "gevent",
    "emmet_core": "emmet-core",
    "tiled": "tiled",
    "redis": "redis",
}

print("\nInstalled package versions:")
for import_name, dist_name in PACKAGE_NAME_MAP.items():
    try:
        version = md.version(dist_name)
        print(f"  - {dist_name}: {version}")
    except md.PackageNotFoundError:
        print(f"  - {dist_name}: NOT INSTALLED")


Running Python: 3.11.15

Installed package versions:
  - crystal-toolkit: 2026.1.22.post0
  - dgl: 2.1.0
  - dash: 3.4.0
  - lightning: 2.6.1
  - matgl: 0.8.5
  - mp-api: 0.41.2
  - numpy: 1.26.4
  - pymatgen: 2024.11.13
  - torch: 2.2.0
  - torchdata: 0.7.1
  - gunicorn: 25.1.0
  - gevent: 25.9.1
  - emmet-core: 0.84.3rc0
  - tiled: 0.2.8
  - redis: 7.3.0


In [3]:
from __future__ import annotations

# Standard-library imports used for dynamic imports, JSON formatting,
# environment variables, and filesystem path handling.
import importlib
import json
import os
import sys
from pathlib import Path
from typing import Any

# Materials Project client used to fetch crystal structures by material ID.
from mp_api.client import MPRester
from pymatgen.core import Structure


/home/sairam/miniforge3/envs/LightshowAI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
def add_repo_to_path(repo_path: str) -> None:
    """Add a repository path to sys.path so local modules can be imported.

    This is optional in this notebook, but can be useful if you have a local
    clone of a project and want to import Python modules from it.
    """
    repo = str(Path(repo_path).resolve())
    if repo not in sys.path:
        sys.path.insert(0, repo)


def fetch_structure(material_id: str, api_key: str) -> Structure:
    """Fetch a pymatgen Structure object from Materials Project.

    Parameters
    ----------
    material_id
        A Materials Project ID such as 'mp-390'.
    api_key
        Your Materials Project API key.

    Notes
    -----
    You can either pass the key directly or store it in the environment as
    MP_API_KEY and read it in the configuration cell below.
    """
    if not api_key:
        raise ValueError(
            "No Materials Project API key found. Set MP_API_KEY or assign MP_API_KEY in the config cell."
        )

    with MPRester(api_key) as mpr:
        structure = mpr.get_structure_by_material_id(material_id)

    if structure is None:
        raise RuntimeError(f"No structure returned for material ID: {material_id}")

    return structure


def import_model_module(module_name: str):
    """Import a module by name. Useful for modular notebook workflows."""
    return importlib.import_module(module_name)


def get_callable(module, fn_name: str):
    """Safely retrieve a callable from a Python module."""
    try:
        fn = getattr(module, fn_name)
    except AttributeError as exc:
        raise AttributeError(
            f"Module '{module.__name__}' does not define function '{fn_name}'"
        ) from exc

    if not callable(fn):
        raise TypeError(
            f"Attribute '{fn_name}' in module '{module.__name__}' is not callable"
        )
    return fn


def structure_summary(structure: Structure) -> dict[str, Any]:
    """Return a JSON-friendly summary of a pymatgen structure.

    This is helpful for quick inspection in notebooks because the full
    Structure object contains richer methods and metadata than a simple print.
    """
    return {
        "formula": structure.composition.reduced_formula,
        "num_sites": len(structure),
        "lattice": structure.lattice.as_dict(),
        "species": [str(site.specie) for site in structure],
        "cart_coords": [list(map(float, site.coords)) for site in structure],
        "frac_coords": [list(map(float, site.frac_coords)) for site in structure],
    }


def to_jsonable(obj: Any) -> Any:
    """Convert common Python objects to values that json.dumps can print."""
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]
    if hasattr(obj, "as_dict"):
        try:
            return obj.as_dict()
        except Exception:
            pass
    if hasattr(obj, "__dict__"):
        try:
            return {
                k: to_jsonable(v)
                for k, v in vars(obj).items()
                if not k.startswith("_")
            }
        except Exception:
            pass
    return repr(obj)


def print_block(title: str, obj: Any, pretty: bool = False) -> None:
    """Pretty-print structured content with a section heading."""
    print(f"\n=== {title} ===")
    payload = to_jsonable(obj)
    if pretty:
        print(json.dumps(payload, indent=2, sort_keys=False))
    else:
        print(json.dumps(payload))


## Configure your input material and API key

Update the material ID and provide your Materials Project API key.

### Options for the API key

- Paste it directly into `MP_API_KEY` below.
- Or keep `MP_API_KEY = os.getenv("MP_API_KEY", "")` so the notebook reads it from your environment.

Using an environment variable is safer because it avoids saving the key inside the notebook file.


In [5]:
# Materials Project material identifier. Examples look like mp-390, mp-149, etc.
MATERIAL_ID = "mp-390"

# Preferred option: read your API key from an environment variable.
# In a terminal before launching Jupyter, you can run:
#   export MP_API_KEY="your_api_key_here"
# You can also paste the key directly as a string, but avoid committing it to git.
MP_API_KEY = os.getenv("MP_API_KEY", "EPmd3hM80TPdrhTvsI9TUUPwdqFRD3uk")

# Set PRETTY=True for readable JSON-like output in the notebook.
PRETTY = True

print(f"Fetching structure for {MATERIAL_ID} from Materials Project...")
structure = fetch_structure(MATERIAL_ID, MP_API_KEY)
print_block("STRUCTURE SUMMARY", structure_summary(structure), pretty=PRETTY)


Fetching structure for mp-390 from Materials Project...


Retrieving MaterialsDoc documents: 100%|██████████| 1/1 [00:00<00:00, 14873.42it/s]


=== STRUCTURE SUMMARY ===
{
  "formula": "TiO2",
  "num_sites": 6,
  "lattice": {
    "@module": "pymatgen.core.lattice",
    "@class": "Lattice",
    "matrix": [
      [
        3.54771631,
        0.0,
        -1.31198863
      ],
      [
        -0.48518786,
        3.51438277,
        -1.31198863
      ],
      [
        0.01802498,
        0.02068314,
        5.50138299
      ]
    ],
    "pbc": [
      true,
      true,
      true
    ]
  },
  "species": [
    "Ti",
    "Ti",
    "O",
    "O",
    "O",
    "O"
  ],
  "cart_coords": [
    [
      2.8055156037500004,
      2.20166001625,
      -0.5926371975000002
    ],
    [
      0.27503782625,
      1.33340589375,
      3.4700429274999998
    ],
    [
      0.8988914949374885,
      2.0493055001875304,
      -0.2956574993344864
    ],
    [
      -0.1003220575625115,
      2.9382427626875303,
      2.4550339956655134
    ],
    [
      3.180875487562512,
      0.5968231473124699,
      0.42237173433448616
    ],
    [
      2.1

## Download model checkpoints and define the prediction code

This cell does three things:

1. Creates a local `model_checkpoints/` folder if it does not exist.
2. Downloads the LightshowAI checkpoint files from GitHub if they are missing.
3. Defines the helper classes used to featurize the structure and run prediction.

The downloads happen only once unless you set `overwrite=True` in `ensure_model_checkpoints()`.


In [1]:
import pathlib
import urllib.request
from functools import cache
from typing import List

import numpy as np
import torch
from lightning import LightningModule
from matgl import load_model
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph.compute import (
    compute_pair_vector_and_distance,
    compute_theta_and_phi,
    create_line_graph,
)
from matgl.utils.cutoff import polynomial_cutoff
from pymatgen.core import Structure as PymatgenStructure
from torch import nn

# Base location of the published LightshowAI checkpoint files.
# These are downloaded lazily into the local notebook working directory.
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/AI-multimodal/LightshowAI/main/model_checkpoints"

# Notebook-safe working directory.
# If you started Jupyter in a project folder, the model_checkpoints directory
# will be created there.
PARENT_DIRECTORY = pathlib.Path.cwd().resolve()
MODEL_CHECKPOINTS_PATH = PARENT_DIRECTORY / "model_checkpoints"
XASBLOCKS_PATH = MODEL_CHECKPOINTS_PATH / "xasblock" / "v1.1.1"
M3GNET_PATH = MODEL_CHECKPOINTS_PATH / "M3GNet-MP-2021.2.8-PES"

# Available XAS block checkpoints released in the LightshowAI repository.
# The file naming convention is <ELEMENT>_<THEORY>.ckpt
XASBLOCK_FILES = [
    "Co_FEFF.ckpt",
    "Cr_FEFF.ckpt",
    "Cu_FEFF.ckpt",
    "Cu_VASP.ckpt",
    "Fe_FEFF.ckpt",
    "Mn_FEFF.ckpt",
    "Ni_FEFF.ckpt",
    "Ti_FEFF.ckpt",
    "Ti_VASP.ckpt",
    "V_FEFF.ckpt",
]

# Files required to load the M3GNet backbone used for structure featurization.
M3GNET_FILES = [
    "LICENSE",
    "README.md",
    "model.json",
    "model.pt",
    "state.pt",
]


def _download_file(url: str, destination: pathlib.Path, overwrite: bool = False):
    """Download one file if it is missing locally."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and not overwrite:
        return
    print(f"Downloading {destination.name} ...")
    urllib.request.urlretrieve(url, destination)


def ensure_model_checkpoints(overwrite: bool = False):
    """Create model folders and download required checkpoint files.

    Run this once at the beginning of a session. It is safe to call multiple
    times because existing files are skipped unless overwrite=True.
    """
    XASBLOCKS_PATH.mkdir(parents=True, exist_ok=True)
    M3GNET_PATH.mkdir(parents=True, exist_ok=True)

    for filename in XASBLOCK_FILES:
        url = f"{GITHUB_RAW_BASE}/xasblock/v1.1.1/{filename}"
        dest = XASBLOCKS_PATH / filename
        _download_file(url, dest, overwrite=overwrite)

    for filename in M3GNET_FILES:
        url = f"{GITHUB_RAW_BASE}/M3GNet-MP-2021.2.8-PES/{filename}"
        dest = M3GNET_PATH / filename
        _download_file(url, dest, overwrite=overwrite)


# Ensure checkpoints are present before any model code runs.
ensure_model_checkpoints()

# Useful for checking which element/theory combinations are available locally.
AVAILABLE_COMBINATIONS = sorted(f.stem for f in XASBLOCKS_PATH.glob("*.ckpt"))
print("Available checkpoint combinations:", AVAILABLE_COMBINATIONS)


class XASBlock(nn.Sequential):
    """Simple feed-forward block used by the released checkpoint."""
    DROPOUT = 0.5

    def __init__(self, input_dim: int, hidden_dims: List[int], output_dim: int):
        dims = [input_dim] + hidden_dims + [output_dim]
        layers = []
        for i, (w1, w2) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(w1, w2))
            if i < len(dims) - 2:
                layers.append(nn.BatchNorm1d(w2))
                layers.append(nn.SiLU())
                layers.append(nn.Dropout(self.DROPOUT))
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


class XASBlockModule(LightningModule):
    """Lightning wrapper around the XAS block checkpoint."""
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    @classmethod
    def load(
        cls,
        element: str,
        spectroscopy_type: str,
        pattern=XASBLOCKS_PATH / "{element}_{type}.ckpt",
    ):
        # Make sure files are present before attempting to load a checkpoint.
        ensure_model_checkpoints()
        pattern = str(pattern)
        path = pattern.format(element=element, type=spectroscopy_type)

        if not pathlib.Path(path).exists():
            raise FileNotFoundError(
                f"Checkpoint not found: {path}\n"
                f"Available combinations: {AVAILABLE_COMBINATIONS}"
            )

        # These dimensions match the published v1.1.1 checkpoint layout.
        model = XASBlock(
            input_dim=64,
            hidden_dims=[500, 500, 550],
            output_dim=141,
        )
        module = cls.load_from_checkpoint(checkpoint_path=path, model=model)
        return module


class M3GNetFeaturizer:
    """Convert a pymatgen Structure into learned node features using M3GNet."""
    def __init__(self, model=None, n_blocks=None):
        self.model = model or M3GNetFeaturizer._load_m3gnet()
        self.model.eval()
        self.n_blocks = n_blocks or self.model.n_blocks

    def featurize(
        self,
        structure: PymatgenStructure,
    ):
        graph_converter = Structure2Graph(
            self.model.element_types, self.model.cutoff
        )
        g, state_attr = graph_converter.get_graph(structure)

        node_types = g.ndata["node_type"]
        bond_vec, bond_dist = compute_pair_vector_and_distance(g)

        g.edata["bond_vec"] = bond_vec.to(g.device)
        g.edata["bond_dist"] = bond_dist.to(g.device)

        with torch.no_grad():
            expanded_dists = self.model.bond_expansion(g.edata["bond_dist"])

            l_g = create_line_graph(g, self.model.threebody_cutoff)

            l_g.apply_edges(compute_theta_and_phi)
            g.edata["rbf"] = expanded_dists
            three_body_basis = self.model.basis_expansion(l_g)
            three_body_cutoff = polynomial_cutoff(
                g.edata["bond_dist"], self.model.threebody_cutoff
            )
            node_feat, edge_feat, state_feat = self.model.embedding(
                node_types, g.edata["rbf"], state_attr
            )

            for i in range(self.n_blocks):
                edge_feat = self.model.three_body_interactions[i](
                    g,
                    l_g,
                    three_body_basis,
                    three_body_cutoff,
                    node_feat,
                    edge_feat,
                )
                edge_feat, node_feat, state_feat = self.model.graph_layers[i](
                    g, edge_feat, node_feat, state_feat
                )

        # Move features to CPU before converting to NumPy for notebook use.
        res = np.array(node_feat.detach().cpu().numpy())
        return res

    @cache
    @staticmethod
    def _load_m3gnet(path=M3GNET_PATH):
        """Load and cache the M3GNet model so repeated predictions are faster."""
        ensure_model_checkpoints()
        model = load_model(path).model
        model.eval()
        return model


class XASModel:
    """High-level wrapper that featurizes a structure and predicts a spectrum."""
    featurizer = M3GNetFeaturizer()

    def __init__(self, element: str, spectroscopy_type: str):
        self.element = element
        self.spectroscopy_type = spectroscopy_type
        self.model = XASBlockModule.load(
            element=element, spectroscopy_type=spectroscopy_type
        )
        self.model.eval()

    def _get_feature(self, structure: PymatgenStructure):
        return self.featurizer.featurize(structure)

    def predict(
        self,
        structure: PymatgenStructure,
    ):
        with torch.no_grad():
            feature = self._get_feature(structure)

            # The released notebook code scales the feature vector by 1000
            # before feeding it into the XAS block.
            feature = feature * 1000.0

            device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
            feature = torch.tensor(feature, device=device)
            spectrum = self.model(feature)

        spectrum = spectrum.detach().cpu().numpy().squeeze()
        return spectrum


def predict(structure, absorbing_site, spectroscopy_type):
    """Predict spectra for a given absorbing element in the input structure.

    Parameters
    ----------
    structure
        pymatgen Structure object.
    absorbing_site
        Element symbol such as 'Ti' or 'Fe'.
    spectroscopy_type
        Theory label that matches a downloaded checkpoint, such as 'FEFF' or 'VASP'.
    """
    site_idxs = [
        ii
        for ii, site in enumerate(structure.sites)
        if site.specie.symbol == absorbing_site
    ]
    if len(site_idxs) == 0:
        raise ValueError(
            f"element {absorbing_site} not found in provided structure"
        )

    spec = XASModel(
        element=absorbing_site, spectroscopy_type=spectroscopy_type
    ).predict(structure)

    result = {ii: spec[ii] for ii in site_idxs}
    return result


ModuleNotFoundError: No module named 'torch'

## Run a prediction

Choose an absorbing element and theory that exist in `AVAILABLE_COMBINATIONS`.

Examples from the downloaded checkpoints include `Ti_VASP`, `Ti_FEFF`, `Fe_FEFF`, and `Cu_VASP`.


In [7]:
# Example prediction: titanium using the VASP-trained checkpoint.
# Make sure the element/theory pair exists in AVAILABLE_COMBINATIONS.
element = "Ti"
theory = "VASP"

output = predict(structure, element, theory)
print_block("PREDICTION OUTPUT", output, pretty=True)

specs_array = np.array(list(output.values()))
predicted_spectrum = specs_array.mean(axis=0)
print_block("PREDICTED SPECTRUM (AVERAGED OVER SITES)", predicted_spectrum, pretty=True)



=== PREDICTION OUTPUT ===
{
  "0": "array([0.02362889, 0.02362992, 0.02363205, 0.02363587, 0.02364374,\n       0.0236784 , 0.02372479, 0.02378211, 0.02383053, 0.02385101,\n       0.02385839, 0.02391419, 0.02419167, 0.02491961, 0.02619175,\n       0.02765366, 0.02903827, 0.02987733, 0.03069442, 0.0331122 ,\n       0.03772895, 0.0441726 , 0.05351771, 0.06790829, 0.08948899,\n       0.12060843, 0.1546817 , 0.17609945, 0.19239151, 0.2134919 ,\n       0.23731533, 0.25572243, 0.2620323 , 0.2629814 , 0.27234355,\n       0.28267223, 0.27829903, 0.264795  , 0.2566204 , 0.25712892,\n       0.26029456, 0.2606097 , 0.25488234, 0.2401852 , 0.21689832,\n       0.19300379, 0.175157  , 0.16493124, 0.16103409, 0.16204946,\n       0.16755633, 0.17870279, 0.19360235, 0.20971845, 0.2276461 ,\n       0.24893399, 0.27324548, 0.3010008 , 0.33265075, 0.36526546,\n       0.39628288, 0.42684197, 0.46245596, 0.5034264 , 0.5463719 ,\n       0.5894816 , 0.63235164, 0.67480004, 0.7175982 , 0.76099885,\n       0.80

In [18]:
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient
import torch
import torch.nn as nn
from torchvision import datasets, transforms
import os
import urllib3
from urllib3.exceptions import InsecureRequestWarning

# --- Security Configuration ---
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
urllib3.disable_warnings(InsecureRequestWarning)

AMSC_API_KEY_ENV = "AgG4qPXXX82mJlpb2g2ogBq0lbkeyy66BVlJWyqzwPnnN16gVXt7CBg494Nv1jbEg9q4yQd5owMNkOu9WBDbnCGoO2VfpYXvVHyD3V1"   # token lives here

if AMSC_API_KEY_ENV not in os.environ:
    print(f"Warning: {AMSC_API_KEY_ENV} not found in environment.")

# Function to inject API Key (Reuse your existing logic here)
def enable_amsc_x_api_key():
    import mlflow.utils.rest_utils as rest_utils
    api_key = os.environ.get("AM_SC_API_KEY")
    if api_key:
        _orig = rest_utils.http_request
        def patched(host_creds, endpoint, method, *args, **kwargs):
            h = dict(kwargs.get("headers") or kwargs.get("extra_headers") or {})
            h["X-Api-Key"] = api_key
            kwargs["headers" if "headers" in kwargs else "extra_headers"] = h
            return _orig(host_creds, endpoint, method, *args, **kwargs)
        rest_utils.http_request = patched

enable_amsc_x_api_key()
mlflow.set_tracking_uri("https://mlflow.american-science-cloud.org")



def upload_local_model():
    model = XASModel(
        element='Ti', spectroscopy_type='VASP'
    )
    # 2. Load the local weights
    model.load_state_dict(torch.load("/home/sairam/LightshowAI/LightshowAI/tiled_intregation/model_checkpoints/xasblock/v1.1.1/V_FEFF.ckpt"))
    model.eval()

    mlflow.set_experiment("xas_ti_vasp")

    # 3. Log and Register in one go
    with mlflow.start_run(run_name="Manual_Upload") as run:
        print(f"Uploading local model to MLflow...")
        
        # We use log_model to package the weights + environment together
        mlflow.pytorch.log_model(
            pytorch_model=model,
            artifact_path="model",
            registered_model_name="XAS_Ti_VASP" # It will create a new version here
        )
        print(f"✅ Successfully uploaded! Run ID: {run.info.run_id}")

if __name__ == "__main__":
    upload_local_model()



AttributeError: partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

In [16]:
!pip uninstall torch torchvision -y

Found existing installation: torchvision 0.26.0
Uninstalling torchvision-0.26.0:
  Successfully uninstalled torchvision-0.26.0


In [17]:
pip install torch==2.2.2 torchvision==0.17.2

  Using cached triton-2.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.6/755.6 MB 35.1 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 36.0 MB/s  0:00:00
Using cached triton-2.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (167.9 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:━━━━━━━━━━━━━━━━━━ 0/3 [triton]
      Successfully uninstalled triton-3.6.0━━━━━ 0/3 [triton]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision] [torchvision]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
lightshowai 0.0.1 requires torch==2.2.0, but you have torch 2.2.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [1]:
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor

# set the experiment id
mlflow.set_experiment(experiment_id="80")

mlflow.autolog()
db = load_diabetes()

X_train, X_test, y_train, y_test = train_test_split(db.data, db.target)

# Create and train models.
rf = RandomForestRegressor(n_estimators=100, max_depth=6, max_features=3)
rf.fit(X_train, y_train)

# Use the model to make predictions on the test dataset.
predictions = rf.predict(X_test)

ModuleNotFoundError: No module named 'mlflow'